# Optuna Tuning: Intermediate

This notebook builds on the basics and covers:
1. **Pre-built config spaces** (`build_nsgaii_config_space`) instead of manual `ParamSpace`
2. **Multi-problem tuning** across several ZDT instances
3. **Comparing Optuna samplers**: TPE vs CMA-ES vs Random
4. **Parallel execution** with `n_jobs`
5. **Saving and analyzing history**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vamos import optimize
from vamos.foundation.quality_indicators import compute_hypervolume
from vamos.engine.tuning import (
    ModelBasedTuner,
    TuningTask,
    Instance,
    EvalContext,
    build_nsgaii_config_space,
    config_from_assignment,
    save_history_csv,
)

plt.style.use("ggplot")
print("Imports OK")

## 1. Pre-built Config Space

Instead of defining parameters manually, VAMOS provides `build_nsgaii_config_space()` which includes all relevant NSGA-II hyperparameters with conditional dependencies.

In [ ]:
config_space = build_nsgaii_config_space()
param_space = config_space.to_param_space()

print(f"Parameters ({len(param_space.params)}):")
for name, p in param_space.params.items():
    print(f"  {name}: {type(p).__name__}", end="")
    if hasattr(p, "low"):
        print(f" [{p.low}, {p.high}]", end="")
    if hasattr(p, "choices"):
        print(f" {list(p.choices)}", end="")
    print()

if param_space.conditions:
    print(f"\nConditional params: {len(param_space.conditions)}")

## 2. Multi-Problem Tuning Task

We tune across **multiple ZDT problems** simultaneously. The tuner evaluates each config on all instances × seeds and aggregates the scores.

In [ ]:
REF_POINT = np.array([1.1, 1.1])
N_VAR = 30
BUDGET = 5000  # FEs per run

instances = [
    Instance(name="zdt1", n_var=N_VAR, kwargs={}),
    Instance(name="zdt2", n_var=N_VAR, kwargs={}),
    Instance(name="zdt4", n_var=N_VAR, kwargs={}),
]


def eval_fn(config: dict, ctx: EvalContext) -> float:
    """Evaluate NSGA-II on the given instance."""
    if "pop_size" not in config:
        config["pop_size"] = 100
    cfg = config_from_assignment("nsgaii", config)

    result = optimize(
        ctx.instance.name,
        algorithm="nsgaii",
        algorithm_config=cfg,
        max_evaluations=ctx.budget,
        seed=ctx.seed,
        n_var=ctx.instance.n_var,
        engine="numpy",
    )

    if result.F is None or len(result.F) == 0:
        return 0.0
    return compute_hypervolume(result.F, REF_POINT)


task = TuningTask(
    name="multi_problem_nsgaii",
    param_space=param_space,
    instances=instances,
    seeds=[0, 1],                # 2 seeds (fast demo)
    budget_per_run=BUDGET,
    maximize=True,
    aggregator=lambda scores: float(np.mean(scores)),
)

print(f"Each trial evaluates: {len(instances)} instances x {len(task.seeds)} seeds = {len(instances) * len(task.seeds)} MOEA runs")
print(f"Each MOEA run: {BUDGET} FEs")

## 3. Compare Optuna Samplers

We compare three samplers:
- **TPE**: Tree-structured Parzen Estimator (default, good general-purpose)
- **CMA-ES**: Covariance Matrix Adaptation (good for continuous params)
- **Random**: Baseline random search

Available samplers: `tpe`, `cmaes`, `random`, `nsgaii`, `nsgaiii`, `qmc`, `gp`

In [ ]:
MAX_TRIALS = 15  # configs per sampler (small for demo)
SAMPLERS = ["tpe", "cmaes", "random"]

results = {}

for sampler_name in SAMPLERS:
    print(f"\n--- Running sampler: {sampler_name} ---")

    tuner = ModelBasedTuner(
        task=task,
        max_trials=MAX_TRIALS,
        backend="optuna",
        optuna_sampler=sampler_name,
        seed=42,
        n_jobs=1,
    )

    best_config, history = tuner.run(eval_fn)
    scores = [t.score for t in history]
    results[sampler_name] = {
        "best_config": best_config,
        "history": history,
        "scores": scores,
        "best": max(scores),
    }
    print(f"  Best HV: {max(scores):.6f}  |  Mean: {np.mean(scores):.6f}")

print("\nAll samplers done!")

## 4. Visualize Sampler Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence curves
for name, res in results.items():
    best_so_far = np.maximum.accumulate(res["scores"])
    axes[0].plot(best_so_far, "o-", label=f"{name} (best={res['best']:.4f})")

axes[0].set_xlabel("Trial")
axes[0].set_ylabel("Best HV so far")
axes[0].set_title("Sampler Convergence")
axes[0].legend()

# Box plot
data = [res["scores"] for res in results.values()]
bp = axes[1].boxplot(data, labels=list(results.keys()), patch_artist=True)
colors = ["#4C72B0", "#DD8452", "#55A868"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
axes[1].set_ylabel("Hypervolume")
axes[1].set_title("Score Distribution by Sampler")

plt.tight_layout()
plt.show()

## 5. Parallel Execution

Set `n_jobs > 1` to evaluate multiple trials in parallel. Optuna handles thread-safe trial sampling.

In [ ]:
import time

# Sequential
t0 = time.perf_counter()
tuner_seq = ModelBasedTuner(
    task=task, max_trials=10, backend="optuna",
    optuna_sampler="tpe", seed=0, n_jobs=1,
)
_, hist_seq = tuner_seq.run(eval_fn)
t_seq = time.perf_counter() - t0

# Parallel (2 jobs)
t0 = time.perf_counter()
tuner_par = ModelBasedTuner(
    task=task, max_trials=10, backend="optuna",
    optuna_sampler="tpe", seed=0, n_jobs=2,
)
_, hist_par = tuner_par.run(eval_fn)
t_par = time.perf_counter() - t0

print(f"Sequential (n_jobs=1): {t_seq:.1f}s")
print(f"Parallel   (n_jobs=2): {t_par:.1f}s")
print(f"Speedup: {t_seq / t_par:.2f}x")

## 6. Save History to CSV

In [ ]:
# Pick the best sampler's history
best_sampler = max(results, key=lambda s: results[s]["best"])
best_history = results[best_sampler]["history"]

save_history_csv(best_history, "tuning_results.csv")
print(f"Saved {len(best_history)} trials to tuning_results.csv")

# Quick peek
df = pd.DataFrame([
    {"trial": t.trial_id, "score": t.score, **t.config}
    for t in best_history
])
df.head(10)

## 7. Parameter Importance (quick analysis)

Which parameters had the most impact on the score?

In [ ]:
# Correlation between each param and score
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ("trial", "score")]

if numeric_cols:
    correlations = {col: df["score"].corr(df[col]) for col in numeric_cols}
    corr_series = pd.Series(correlations).sort_values(key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(8, max(3, len(corr_series) * 0.4)))
    colors = ["#4C72B0" if v > 0 else "#C44E52" for v in corr_series]
    corr_series.plot.barh(ax=ax, color=colors)
    ax.set_xlabel("Correlation with HV")
    ax.set_title("Parameter-Score Correlation")
    ax.axvline(0, color="black", lw=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric parameters to analyze.")

## Summary

| Feature | How |
|---|---|
| Pre-built config space | `build_nsgaii_config_space().to_param_space()` |
| Multi-problem tuning | Pass multiple `Instance` objects |
| Compare samplers | Change `optuna_sampler` (`tpe`, `cmaes`, `random`, ...) |
| Parallel trials | Set `n_jobs > 1` |
| Save results | `save_history_csv(history, path)` |

**Next**: See `33_optuna_tuning_advanced.ipynb` for multi-fidelity pruning, persistent storage, and budget-level strategies.